# CityOps Copilot — Agent Memory Workshop

An inspector / city-operations copilot that remembers every asset's history across inspectors. By the end, a new inspector will encounter a corrosion concern on Harbor Bridge and the copilot will already know what the prior inspector observed, recommended, and graded — without anyone telling it.

Built on Oracle AI Database + the `oracleagentmemory` SDK, seeded with a realistic dataset of 308 maintenance logs and ~220 inspection findings across 26 urban infrastructure assets (bridges, substations, pipelines, water treatment, sensors, comms towers, seawalls).

>  Open `docs/overview.md` for the one-pager.

## What You'll Build

An agent that uses **four memory layers** simultaneously:

| Layer | What it stores | How it's populated |
|---|---|---|
| Auto-extracted `fact` / `preference` / `guideline` (SDK) | Inspector tribal knowledge from maintenance narratives | SDK's extractor runs on every event report |
| `CITY_ASSET` (hand-rolled SQL) | The plant's asset registry — 26 assets from real data | Bulk-loaded from `data/maintenance_logs.json` + `data/inspection_reports.json` |
| `CITY_INSPECTION_FINDING` (hand-rolled SQL with `VECTOR(384)`) | Structured findings — category, severity, description, recommendation, grade | ~220 findings bulk-loaded from `data/inspection_reports.json` |
| Conversational messages (SDK) | Per-asset inspection threads, naturally scoped to the asset | `thread.add_messages` |

>  **Two storage layers?** The SDK only allows 4 native record types (`fact`, `memory`, `preference`, `guideline`). Domain objects need their own SQL tables. Oracle's converged engine makes this clean — same connection, vector search via `VECTOR_DISTANCE()` available everywhere. See `docs/sdk-data-model.md`.

## Select Your Kernel

**Step 1.** Click **Select Kernel** in the top-right.

**Step 2.** Choose **CityOps Copilot** (or Python 3.11).

If you don't see the kernel, wait for the Codespace setup to complete.

In [ ]:
# Dependencies are pre-installed in this Codespace via .devcontainer/setup.sh
# If you are running locally, uncomment the next line:
# ! pip install -qU oracleagentmemory sentence-transformers oracledb openai matplotlib

# Part 1: Oracle Setup

Oracle AI Database is reachable in one of two ways:

- **Codespaces (default):** Oracle Free runs as a Docker service alongside the dev container — `localhost:1521`, service `FREEPDB1`, user `VECTOR`.
- **Local dev against ADB:** drop a `.env` file in the project root with your wallet path and ADB credentials; the next cell loads it.

Either way, the SDK creates its tables under prefix `CITY_` in Part 2.

In [ ]:
# Load environment from a project-root .env (no-op if missing — Codespaces path).
# Keeps the notebook portable across local-dev (ADB+wallet), Codespaces (local Oracle
# Free), and headless nbconvert runs without depending on the VS Code Python extension.
import os
from pathlib import Path

_env_file = next((p for p in (Path(".env"), Path("../.env")) if p.exists()), None)
if _env_file is not None:
    for _line in _env_file.read_text().splitlines():
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _k, _v = _line.split("=", 1)
        # .setdefault — env vars already set externally (Codespaces secrets) win.
        os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
    print(f" Loaded env vars from {_env_file.resolve()}")
else:
    print("ℹ No .env file found — assuming env vars are set externally")

if os.environ.get("ORACLE_WALLET_LOCATION"):
    print(f"  mode: ADB / wallet  ({os.environ.get('ORACLE_DSN','<DSN missing>')[:60]}...)")
else:
    print("  mode: local (Codespaces / Oracle Free on localhost:1521)")

In [ ]:
import oracledb, os, time

def connect_to_oracle(max_retries=3, retry_delay=5,
                     program="cityops_copilot"):
    """Connect to Oracle (ADB+wallet if ORACLE_WALLET_LOCATION set, else local)."""
    wallet_dir = os.environ.get("ORACLE_WALLET_LOCATION")
    if wallet_dir:
        kwargs = dict(
            user=os.environ["ORACLE_USER"],
            password=os.environ["ORACLE_PASSWORD"],
            dsn=os.environ["ORACLE_DSN"],
            config_dir=wallet_dir,
            wallet_location=wallet_dir,
            wallet_password=os.environ.get("ORACLE_WALLET_PASSWORD", ""),
            program=program,
        )
        mode = f"ADB ({os.environ['ORACLE_DSN'][:50]}...)"
    else:
        kwargs = dict(
            user=os.environ.get("ORACLE_USER", "VECTOR"),
            password=os.environ.get("ORACLE_PASSWORD", "VectorPwd_2025"),
            dsn=os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1"),
            program=program,
        )
        mode = f"local ({kwargs['dsn']})"

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries} — {mode}...")
            conn = oracledb.connect(**kwargs)
            print(f" Connected as {kwargs['user']}.")
            return conn
        except oracledb.OperationalError as e:
            print(f" {e}")
            if attempt < max_retries:
                time.sleep(retry_delay)
            else:
                raise
    raise ConnectionError("Could not connect.")

In [ ]:
vector_conn = connect_to_oracle()
print("Using user:", vector_conn.username)

 Connected. Next: wire up the SDK's embedder + create the schema.

>  **Key insight — Part 1:** Oracle AI Database is a converged engine. The SDK's tables, our hand-rolled `CITY_ASSET` / `CITY_INSPECTION_FINDING` tables, and any other SQL all live on this same connection. Vector search via `VECTOR_DISTANCE()` works across all of them.

###  Optional: Reset The Workspace

If you've run this workshop before (or are sharing an Oracle instance with someone who has), the SDK's tables (`CITY_THREAD`, `CITY_MESSAGE`, `CITY_MEMORY`, `CITY_RECORD_CHUNKS`, `CITY_ACTOR_PROFILE`) and the workshop's custom tables (`CITY_ASSET`, `CITY_INSPECTION_FINDING`) may still hold state from prior runs.

Why this matters: when you call `report_event("smoke_thread", ...)` in TODO 2, the SDK does `get_thread("smoke_thread")` first — and will pick up an existing thread from a previous run, **appending** the new message to whatever's already there instead of starting fresh.

**Run the cell below to wipe everything CITY_*, then re-run from Part 2.** If you've already initialised the SDK (`memory = OracleAgentMemory(...)`) in Part 2, you'll need to **restart the kernel** after this reset — the in-memory `memory` and `thread` objects still reference the dropped tables.

Skip this cell on the first-ever run.

In [ ]:
# Optional: wipe all CITY_* tables for a clean slate.
# Safe to run multiple times; safe to skip.
_to_drop = (
    # Drop the hand-rolled tables first (they have FKs into CITY_ASSET).
    "CITY_INSPECTION_FINDING", "CITY_ASSET",
    # Then the SDK's tables (FK-cascaded in this order).
    "CITY_MEMORY", "CITY_MESSAGE", "CITY_RECORD_CHUNKS",
    "CITY_THREAD", "CITY_ACTOR_PROFILE",
    "CITY_ORACLEAGENTMEMORY_SCHEMA_META",
)
with vector_conn.cursor() as cur:
    for t in _to_drop:
        try:
            cur.execute(f'DROP TABLE "{t}" CASCADE CONSTRAINTS')
            print(f"  dropped {t}")
        except Exception:
            pass  # table didn't exist — fine
vector_conn.commit()
print("\n Workspace clean. RESTART THE KERNEL before re-running Part 2 if you've already done it.")

# Part 2: The Embedder and SDK

The SDK handles vectorisation and search — but needs an embedder plug-in. You'll provide a local `sentence-transformers` model (384-dim, no API key, no per-call cost).

**TODO 1: Implement `LocalSentenceTransformerEmbedder`.**

1. Subclass `IEmbedder`.
2. Hold a `SentenceTransformer` instance from `"sentence-transformers/all-MiniLM-L6-v2"`.
3. `embed` returns `model.encode(texts, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)`.
4. `embed_async` runs `embed` on a thread via `asyncio.to_thread`.

In [ ]:
from oracleagentmemory.apis.embedders.embedder import IEmbedder
from sentence_transformers import SentenceTransformer
import numpy as np
import asyncio


class LocalSentenceTransformerEmbedder(IEmbedder):
    """Bridges sentence-transformers to the SDK's IEmbedder."""

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        # TODO 1: load the model into self.model
        # YOUR CODE HERE
        pass

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        # TODO 1: encode with normalize_embeddings=True; return float32
        # YOUR CODE HERE
        pass

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        # TODO 1: run self.embed on a thread via asyncio.to_thread
        # YOUR CODE HERE
        pass


In [ ]:
# PASS: Checkpoint: TODO 1
embedder = LocalSentenceTransformerEmbedder()
_v = embedder.embed(["corrosion on bearing assembly"])
assert _v.shape == (1, 384), f"Expected (1, 384), got {_v.shape}"
assert _v.dtype == np.float32, f"Expected float32, got {_v.dtype}"
print("PASS: TODO 1 passed — embedder returns float32 (n, 384) arrays")

### SDK Initialization (Pre-Built)

Wires up `OracleAgentMemory` and the SDK's internal LLM. No changes needed.

- `extract_memories=True` turns on automatic LLM extraction of `fact` / `preference` / `guideline` / `memory` on every `add_messages` call.
- `table_name_prefix="CITY_"` namespaces the SDK's tables.

In [ ]:
import os
from openai import OpenAI
from oracleagentmemory.core import OracleAgentMemory, SchemaPolicy
from oracleagentmemory.core.llms.llm import Llm
from oracleagentmemory.apis.thread import Message

OCI_GENAI_ENDPOINT = os.environ.get(
    "OCI_GENAI_ENDPOINT",
    "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com/openai/v1",
)
OCI_GENAI_API_KEY = os.environ["OCI_GENAI_API_KEY"]

client = OpenAI(base_url=OCI_GENAI_ENDPOINT, api_key=OCI_GENAI_API_KEY)
sdk_llm = Llm(
    model="openai/xai.grok-3-fast",
    api_base=OCI_GENAI_ENDPOINT,
    api_key=OCI_GENAI_API_KEY,
)

memory = OracleAgentMemory(
    connection=vector_conn,
    embedder=embedder,
    llm=sdk_llm,
    extract_memories=True,
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    table_name_prefix="CITY_",
)
print(" OracleAgentMemory ready. Tables under prefix CITY_*")

# Part 3: City Asset Registry + Auto-Extraction

Two pieces in this part:

1. The **`CITY_ASSET` registry** — 26 real urban infrastructure assets, loaded from `data/maintenance_logs.json` + `data/inspection_reports.json`. Hand-rolled SQL table outside the SDK.
2. **Auto-extraction from real maintenance narratives** — the SDK's extractor turns paragraph-long inspection write-ups into typed `fact` / `preference` / `guideline` records inside `CITY_MEMORY`.

Both pre-built cells run automatically; the TODOs are the auto-extraction logic.

In [ ]:
# Pre-built: load the real asset registry from the committed JSON files.
# The 26 assets are the union of asset_name fields across both datasets.
import json
from pathlib import Path

_data_dir = Path("data") if Path("data").exists() else Path("../data")
with open(_data_dir / "maintenance_logs.json") as f:
    _logs = json.load(f)
with open(_data_dir / "inspection_reports.json") as f:
    _reports = json.load(f)

_asset_names = sorted(set(x["asset_name"] for x in _logs) | set(x["asset_name"] for x in _reports))
print(f"Loaded {len(_logs)} maintenance logs, {len(_reports)} inspection reports.")
print(f"Discovered {len(_asset_names)} unique assets.")

In [ ]:
# Pre-built: classify each asset by name heuristics into one of 8 asset classes.
def classify_asset(name: str) -> str:
    n = name.lower()
    if "bridge" in n or "overpass" in n: return "bridge"
    if "substation" in n:                  return "substation"
    if "pipeline" in n:                    return "pipeline"
    if "water" in n or "outfall" in n or "treatment" in n: return "water"
    if "solar" in n or "gas distribution" in n:           return "energy"
    if "sensor" in n or "array" in n or "monitor" in n or "gauge" in n: return "sensor"
    if "tower" in n or "relay" in n:      return "comms"
    if "seawall" in n or "retaining" in n or "booster" in n: return "civil"
    return "other"

equipment = [{"asset_id": name, "asset_class": classify_asset(name)} for name in _asset_names]
from collections import Counter
print("Asset class distribution:", dict(Counter(e['asset_class'] for e in equipment)))
print("\nFirst 5 assets:")
for e in equipment[:5]:
    print(f"  {e['asset_id']:40} -> {e['asset_class']}")

In [ ]:
# Pre-built: create PLANT_ASSET... wait, no — CITY_ASSET — and bulk-INSERT all 26.
with vector_conn.cursor() as cur:
    try:
        cur.execute("DROP TABLE CITY_INSPECTION_FINDING CASCADE CONSTRAINTS")
    except Exception:
        pass
    try:
        cur.execute("DROP TABLE CITY_ASSET CASCADE CONSTRAINTS")
    except Exception:
        pass
    cur.execute("""
        CREATE TABLE CITY_ASSET (
            asset_id     VARCHAR2(128) PRIMARY KEY,
            asset_class  VARCHAR2(32) NOT NULL,
            created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    cur.executemany(
        "INSERT INTO CITY_ASSET (asset_id, asset_class) VALUES (:1, :2)",
        [(e["asset_id"], e["asset_class"]) for e in equipment],
    )
vector_conn.commit()
print(f" Inserted {len(equipment)} rows into CITY_ASSET.")

In [ ]:
# Helper: look up one asset row as a dict.
def get_asset(asset_id: str) -> dict | None:
    with vector_conn.cursor() as cur:
        cur.execute(
            "SELECT asset_id, asset_class FROM CITY_ASSET WHERE asset_id = :id",
            id=asset_id,
        )
        row = cur.fetchone()
    if not row:
        return None
    return {"asset_id": row[0], "asset_class": row[1]}

print(get_asset("Harbor Bridge"))

### Auto-Extraction

When you call `thread.add_messages(...)` with `extract_memories=True`, the SDK runs its extractor LLM internally. It reads the new message + cached running summary + past memories, and outputs typed records (`fact` / `preference` / `guideline` / `memory`) into `CITY_MEMORY`.

**You don't write the extraction prompt.** The SDK does.

**TODO 2: Implement `report_event(asset_id, inspector, narrative, thread_id)`.**

1. **Get-or-create the thread** for this asset:
   - Try `memory.get_thread(thread_id)`.
   - If it raises or returns None, call `memory.create_thread(user_id=inspector, thread_id=thread_id, agent_id="CITY")`.
2. **Build a content string** that includes both asset and inspector — e.g. `f"[Asset: {asset_id}] [Inspector: {inspector}] {narrative}"`.
3. **Call `thread.add_messages([Message(role="user", content=content)])`** and return the list of message IDs.

In [ ]:
def report_event(asset_id: str, inspector: str, narrative: str, thread_id: str) -> list:
    """Persist a maintenance event narrative and trigger SDK auto-extraction."""
    # TODO 2: get-or-create the thread, build the content string,
    # add_messages, and return the IDs.
    # YOUR CODE HERE
    pass


In [ ]:
# PASS: Checkpoint: TODO 2 — smoke test with one real narrative from the dataset
_seed = _logs[0]  # First maintenance log — likely Harbor Bridge routine inspection
ids = report_event(
    asset_id=_seed["asset_name"],
    inspector="smoke_inspector",
    narrative=_seed["narrative"],
    thread_id="smoke_thread",
)
assert ids and len(ids) >= 1, "TODO 2 incomplete — add_messages should return at least one ID"
print(f"PASS: TODO 2 passed — added message(s): {ids}")

### See What Got Persisted

When `report_event` ran, two things happened inside the SDK:

1. **One row INSERTed into `CITY_MESSAGE`** — the raw narrative
2. **The extractor LLM ran** and INSERTed 0+ typed records into `CITY_MEMORY`

The helper below shows both tables. Run it now to see the smoke-test state, then again after TODO 3.

In [ ]:
def peek_sdk_tables(thread_id: str = None) -> None:
    """Print rows from CITY_MESSAGE and CITY_MEMORY (optionally scoped to one thread)."""
    where = "WHERE thread_id = :tid" if thread_id else ""
    bind = {"tid": thread_id} if thread_id else {}
    with vector_conn.cursor() as cur:
        cur.execute(f"""
            SELECT message_role, SUBSTR(content, 1, 100) AS preview, user_id, agent_id
              FROM CITY_MESSAGE {where}
             ORDER BY order_seq
        """, bind)
        msgs = cur.fetchall()
        print(f" CITY_MESSAGE — {len(msgs)} row(s)" + (f" for thread={thread_id}" if thread_id else ""))
        for role, preview, uid, aid in msgs:
            text = preview.read() if hasattr(preview, 'read') else preview
            print(f"   [{role:9}] user={uid or '-':18} agent={aid or '-':6} | {text}")

        cur.execute(f"""
            SELECT memory_type, SUBSTR(content, 1, 100) AS preview, user_id, agent_id
              FROM CITY_MEMORY {where}
             ORDER BY order_seq
        """, bind)
        mems = cur.fetchall()
        print(f"\n CITY_MEMORY — {len(mems)} row(s)" + (f" for thread={thread_id}" if thread_id else ""))
        for mtype, preview, uid, aid in mems:
            text = preview.read() if hasattr(preview, 'read') else preview
            print(f"   [{mtype:10}] user={uid or '-':18} agent={aid or '-':6} | {text}")

peek_sdk_tables(thread_id="smoke_thread")

**TODO 3: Run 8 narratives through `report_event` and inspect what got extracted.**

1. The cell below samples 8 narratives from `data/maintenance_logs.json` spanning different assets and severities.
2. Loop over them and call `report_event(...)` with `thread_id="inspect_demo"`.
3. Query `memory.search` filtered to the four native record types. **You must specify the same `user_id` you wrote with** (`"inspector_demo"`) — the SDK's high-level search rejects `exact_user_match=False`, and the records have `user_id="inspector_demo"` inherited from the thread, so passing `user_id=None` would return zero. Print each extracted record's `record_type` and `content`.

Expect at least 10+ extracted memories across the 8 narratives — the real maintenance logs are dense and the extractor finds plenty of structure.

In [ ]:
# Pre-built helper: stratified sample of 8 narratives.
def sample_narratives(n: int = 8) -> list:
    """Stratified sample: mix of severities, multiple assets."""
    import random
    random.seed(42)  # Reproducible across re-runs
    buckets = {"routine": [], "warning": [], "critical": []}
    for log in _logs:
        buckets.get(log["severity"], []).append(log)
    # Take roughly proportional sample
    n_routine, n_warning, n_critical = max(1, n // 2), max(1, n // 3), max(1, n - n // 2 - n // 3)
    picks = (random.sample(buckets["routine"], min(n_routine, len(buckets["routine"]))) +
             random.sample(buckets["warning"], min(n_warning, len(buckets["warning"]))) +
             random.sample(buckets["critical"], min(n_critical, len(buckets["critical"]))))
    return picks[:n]

narratives = sample_narratives(8)
print("Sampled narratives (asset, severity):")
for n in narratives:
    print(f"  {n['asset_name']:40} [{n['severity']}]")

In [ ]:
# TODO 3:
# 1. Loop over `narratives` and call report_event(...) with thread_id="inspect_demo".
#    Use any plausible inspector_id (e.g. "inspector_demo" — same for all 8).
# 2. Call memory.search across the four native record types, scoped to
#    agent_id="CITY", user_id=None, max_results=30, and print the results.
# YOUR CODE HERE


In [ ]:
# PASS: Checkpoint: TODO 3
# The SDK's high-level search REQUIRES a specific user_id (it rejects
# exact_user_match=False). We wrote the records with inspector="inspector_demo",
# so we search for the same.
_results = memory.search(
    query="recurring asset concerns and inspector practices",
    user_id="inspector_demo",
    agent_id="CITY",
    record_types=["fact", "preference", "guideline", "memory"],
    max_results=50,
)
assert len(_results) >= 5, (
    f"TODO 3 — expected at least 5 extracted records from 8 narratives, got {len(_results)}.\n"
    "Check OCI_GENAI_API_KEY and that you called report_event for all 8 narratives."
)
print(f"PASS: TODO 3 passed — {len(_results)} memories extracted from 8 narratives")

### Peek At The Tables Again — After 8 Narratives

Same helper, this time scoped to the `inspect_demo` thread. You should see 8 user-role messages in `CITY_MESSAGE` and dozens of typed records in `CITY_MEMORY`. Notice that none of the records are custom types — the SDK rejects them. All extracted records are one of the four natives.

In [ ]:
peek_sdk_tables(thread_id="inspect_demo")

# Part 4: Inspection Findings + Similar-Finding Search

An **inspection finding** isn't a `fact` or a `preference` — it's a structured domain object with `category`, `severity`, `description`, `recommendation`, `inspector`, and `overall_grade`. The SDK rejects custom `record_type` values, so findings live in their own hand-rolled `CITY_INSPECTION_FINDING` SQL table with a `VECTOR(384)` column + HNSW index.

Oracle's converged DB lets us mix vector similarity with relational filters (asset, category, severity, time window) in **one SQL statement** — no metadata-filter quirks, full SQL expressiveness.

>  Open `docs/part-4-findings-and-search.md` for the full walk-through.

In [ ]:
# Pre-built: create CITY_INSPECTION_FINDING with VECTOR(384) + HNSW index.
with vector_conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE CITY_INSPECTION_FINDING (
            finding_id      VARCHAR2(64) PRIMARY KEY,
            asset_id        VARCHAR2(128) NOT NULL,
            inspector       VARCHAR2(128),
            overall_grade   VARCHAR2(2),
            category        VARCHAR2(32),
            severity        VARCHAR2(16),
            description     CLOB NOT NULL,
            recommendation  CLOB,
            days_ago        NUMBER,
            embedding       VECTOR(384) NOT NULL,
            created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            CONSTRAINT fk_finding_asset FOREIGN KEY (asset_id) REFERENCES CITY_ASSET(asset_id)
        )
    """)
    try:
        cur.execute("""
            CREATE VECTOR INDEX city_finding_embedding_idx
            ON CITY_INSPECTION_FINDING (embedding)
            ORGANIZATION INMEMORY NEIGHBOR GRAPH
            DISTANCE COSINE
            WITH TARGET ACCURACY 95
            PARAMETERS (TYPE HNSW, M 16, EFCONSTRUCTION 200)
        """)
    except Exception as e:
        print(f"  (skipped HNSW index: {e})")
vector_conn.commit()
print(" CITY_INSPECTION_FINDING created with VECTOR(384) embedding column.")

In [ ]:
# Pre-built: bulk-load all ~220 findings from the inspection reports.
# No LLM extraction needed — findings are already structured. Just embed + INSERT.
import array, uuid

rows = []
for report in _reports:
    asset_id = report["asset_name"]
    inspector = report["inspector"]
    grade = report["overall_grade"]
    days_ago = report["days_ago"]
    for finding in report["findings"]:
        vec = array.array('f', embedder.embed([finding["description"]])[0].tolist())
        rows.append({
            "finding_id":     str(uuid.uuid4())[:12],
            "asset_id":       asset_id,
            "inspector":      inspector,
            "overall_grade":  grade,
            "category":       finding["category"],
            "severity":       finding["severity"],
            "description":    finding["description"],
            "recommendation": finding["recommendation"],
            "days_ago":       days_ago,
            "embedding":      vec,
        })

with vector_conn.cursor() as cur:
    cur.executemany("""
        INSERT INTO CITY_INSPECTION_FINDING
          (finding_id, asset_id, inspector, overall_grade, category, severity,
           description, recommendation, days_ago, embedding)
        VALUES (:finding_id, :asset_id, :inspector, :overall_grade, :category, :severity,
                :description, :recommendation, :days_ago, :embedding)
    """, rows)
vector_conn.commit()
print(f" Inserted {len(rows)} findings into CITY_INSPECTION_FINDING.")

**TODO 4: Implement `log_finding(...)`.**

Signature:
```python
def log_finding(
    asset_id: str,
    inspector: str,
    category: str,
    severity: str,
    description: str,
    recommendation: str = "",
    overall_grade: str = None,
    days_ago: int = 0,
) -> str:
```

Steps:

1. **Generate `finding_id`** — `str(uuid.uuid4())[:12]`.
2. **Compute the embedding** from `description` via `embedder.embed([description])[0].tolist()`. Wrap in `array.array('f', ...)` so `oracledb` binds it as a `VECTOR` (a plain list triggers `ORA-01484`).
3. **INSERT** into `CITY_INSPECTION_FINDING` using named binds.
4. **Commit** and return `finding_id`.

In [ ]:
import array, uuid

def log_finding(
    asset_id: str,
    inspector: str,
    category: str,
    severity: str,
    description: str,
    recommendation: str = "",
    overall_grade: str = None,
    days_ago: int = 0,
) -> str:
    """Persist a new inspection finding into CITY_INSPECTION_FINDING."""
    # TODO 4: generate finding_id, compute embedding, INSERT, commit, return finding_id.
    # YOUR CODE HERE
    pass


In [ ]:
# PASS: Checkpoint: TODO 4
_fid = log_finding(
    asset_id="Harbor Bridge",
    inspector="checkpoint_test",
    category="corrosion",
    severity="medium",
    description="Surface corrosion on pier 2 bearing assemblies; ~25% section loss observed.",
    recommendation="Remove corrosion, apply primer + finish coat within 60 days.",
    overall_grade="C",
    days_ago=0,
)
assert _fid and isinstance(_fid, str), "TODO 4 — should return a finding_id string"

with vector_conn.cursor() as cur:
    cur.execute(
        "SELECT COUNT(*) FROM CITY_INSPECTION_FINDING WHERE inspector = :i",
        i="checkpoint_test",
    )
    n = cur.fetchone()[0]
assert n == 1, "TODO 4 — checkpoint test finding not retrievable"
print(f"PASS: TODO 4 passed — finding_id={_fid}")

**TODO 5: Implement `find_similar_findings(description, asset_id=None, category=None, k=3)`.**

One SQL statement mixes vector similarity with optional asset + category filters:

```sql
SELECT finding_id, asset_id, inspector, overall_grade, category, severity,
       description, recommendation, days_ago,
       VECTOR_DISTANCE(embedding, :q, COSINE) AS score
  FROM CITY_INSPECTION_FINDING
 WHERE (:asset_id IS NULL OR asset_id = :asset_id)
   AND (:category IS NULL OR category = :category)
 ORDER BY score
 FETCH FIRST :k ROWS ONLY
```

Steps:

1. **Embed the query description** wrapped in `array.array('f', ...)`.
2. **Execute the SQL** with binds `q=query_vec`, `asset_id=asset_id`, `category=category`, `k=k`.
3. Return a list of dicts. For `description` and `recommendation` CLOB columns, call `.read()` to materialise them.

In [ ]:
def find_similar_findings(description: str, asset_id: str = None, category: str = None, k: int = 3) -> list:
    """Vector-search CITY_INSPECTION_FINDING, optionally narrowed to one asset and/or category.

    Returns a list of dicts:
        {finding_id, asset_id, inspector, overall_grade, category, severity,
         description, recommendation, days_ago, score}
    """
    # TODO 5: embed the query, run the SQL with VECTOR_DISTANCE, return list of dicts.
    # YOUR CODE HERE
    pass


In [ ]:
# PASS: Checkpoint: TODO 5
_broad = find_similar_findings("bearing corrosion at piers", k=5)
_bridge = find_similar_findings("bearing corrosion at piers", asset_id="Harbor Bridge", k=5)
_corrosion_only = find_similar_findings("bearing corrosion at piers", category="corrosion", k=5)

assert _broad and len(_broad) >= 3, f"TODO 5 — broad search returned {len(_broad) if _broad else 0} hits"
assert _bridge and all(r["asset_id"] == "Harbor Bridge" for r in _bridge), \
    "TODO 5 — asset_id filter should restrict results to Harbor Bridge"
assert _corrosion_only and all(r["category"] == "corrosion" for r in _corrosion_only), \
    "TODO 5 — category filter should restrict results to corrosion only"
print(f"PASS: TODO 5 passed — broad={len(_broad)}, asset-filtered={len(_bridge)}, category-filtered={len(_corrosion_only)}")
for r in _bridge[:3]:
    print(f"  score={r['score']:.3f}  [{r['category']}/{r['severity']}]  {str(r['description'])[:90]}")

# Part 5: Scoping — Inspector vs City

Three scope dimensions: `user_id` (inspector), `agent_id` (`CITY`), `thread_id` (per asset). Enforced as SQL `WHERE` predicates — cross-user leakage is impossible at the DB layer.

**No TODO** — just run the demo cells to see scoping in action.

In [ ]:
# Inspector Mercer writes a personal note (user-scoped, invisible to others)
memory.add_memory(
    content="Remember to swap shifts with Jordan next Tuesday.",
    user_id="Evelyn_H_Mercer",
)

# Mercer also writes a city-wide tribal-knowledge guideline (agent-scoped)
memory.add_memory(
    content="On Harbor Bridge, inspect Pier 2 bearings annually — corrosion-prone since 2024.",
    agent_id="CITY",
)
print(" Mercer wrote one personal memory and one city-wide memory.")

In [ ]:
# Inspector Vance searches at user scope — should NOT see Mercer's personal note
vance_personal = memory.search(
    query="shift swap notes",
    user_id="Jordan_Vance",
    record_types=["memory"],
    max_results=10,
)

# Vance searches at city scope — SHOULD see the Pier 2 guideline
vance_city = memory.search(
    query="Harbor Bridge Pier 2 bearings",
    user_id=None,           # explicitly leave user dimension unconstrained
    agent_id="CITY",
    record_types=["memory"],
    max_results=10,
)

print(f"  Vance's personal-scope hits for 'shift swap': {len(vance_personal)}  (should be 0)")
print(f"  Vance's city-scope hits for 'Pier 2 bearings': {len(vance_city)}  (should be ≥ 1)")

In [ ]:
# Assertions — multi-tenancy is enforced at the SQL layer
assert all("shift swap" not in r.record.content.lower() for r in vance_personal), \
    "Cross-inspector leak: Mercer's personal note showed up in Vance's user-scoped search"
assert any("Pier 2" in r.record.content for r in vance_city), \
    "Pier 2 guideline not retrievable at agent scope — check agent_id wiring"
print(" Multi-tenancy verified: Vance sees city-wide guidelines but NOT Mercer's personal notes.")

>  **Key insight — Part 5:** scoping is enforced as a SQL `WHERE` clause on `user_id` / `agent_id` / `thread_id` columns — not as a soft filter in Python. A bug in your harness can't leak Mercer's note to Vance; only a SQL injection could. For regulated infrastructure, add VPD policies on top (see `docs/part-5-scoping.md`).

# Part 6: The CityOps Copilot — End-to-End

One function, `call_copilot`, ties together: thread resolution (SDK), asset lookup (`CITY_ASSET` SQL), the context card (SDK), similar-finding search (`CITY_INSPECTION_FINDING` SQL via `VECTOR_DISTANCE()`), agent LLM call, and persistence (which triggers auto-extraction).

>  Open `docs/part-6-copilot-end-to-end.md` for the architecture diagram.

In [ ]:
# Pre-built: system prompt + LLM call helper.
COPILOT_SYSTEM_PROMPT = """You are a CityOps inspection copilot.

Each turn you are given:
- The current inspection narrative
- The asset record (class)
- A thread context card (recent inspector messages + extracted facts/guidelines)
- Up to 3 similar past findings on the same asset (with category, severity,
  recommendation, prior inspector, prior overall grade)

Your job: suggest a likely diagnosis or characterisation; cite prior findings
with their inspector + grade + recommendation timeline; surface relevant
guidelines from the thread context. Keep responses ≤ 8 sentences.
When safety- or maintenance-critical guidelines apply, name them."""

def call_openai_chat(messages: list, model: str = "xai.grok-3-fast"):
    return client.chat.completions.create(model=model, messages=messages)

### What Does `get_context_card` Actually Return?

Before you wire it into `call_copilot` (TODO 6), see what the SDK gives you. The cell below grabs the `inspect_demo` thread (8 narratives from Part 3) and prints `card.formatted_content` verbatim.

The card is **XML, not Markdown**. Four blocks: `<summary>`, `<topics>`, `<relevant_information>`, `<recent_messages>`.

In [ ]:
_demo_thread = memory.get_thread("inspect_demo")
_demo_card = _demo_thread.get_context_card(
    fallback_message_count=100,
    max_recent_messages=5,
    max_relevant_results=5,
)
print("=" * 70)
print("Raw context-card output — this is the XML the agent LLM will see:")
print("=" * 70)
print(_demo_card.formatted_content)

**TODO 6: Implement `call_copilot(narrative, inspector_id, thread_id, asset_id)`.**

Step by step:

1. **Resolve thread.** `memory.get_thread(thread_id)`; on failure call `memory.create_thread(user_id=inspector_id, thread_id=thread_id, agent_id="CITY")`.

2. **Asset record.** `get_asset(asset_id)` → dict or None. Format as a short string for the prompt.

3. **Context card.** `thread.get_context_card(fallback_message_count=100, max_recent_messages=10, max_relevant_results=8)`. Use `.formatted_content`.

4. **Similar findings.** `find_similar_findings(narrative, asset_id=asset_id, k=3)`. Format as a string — each row has `score`, `category`, `severity`, `description`, `recommendation`, `inspector`, `overall_grade`, `days_ago`.

5. **Build the context string** with sections `# Current inspection narrative`, `# Asset record`, `# Thread context`, `# Similar past findings`.

6. **Call `call_openai_chat`** with the system prompt + context.

7. **Persist** with `thread.add_messages([Message(role="user", content=...), Message(role="assistant", content=answer)])`.

8. **Return** the assistant's answer.

In [ ]:
def call_copilot(narrative: str, inspector_id: str, thread_id: str, asset_id: str) -> str:
    """End-to-end CityOps copilot turn: build context, query LLM, persist."""
    # TODO 6:
    # 1. Resolve thread (get_thread or create_thread)
    # 2. Lookup asset via get_asset()
    # 3. Build context card via thread.get_context_card
    # 4. Find similar findings via find_similar_findings(asset_id=...)
    # 5. Assemble context string
    # 6. call_openai_chat
    # 7. thread.add_messages with user + assistant
    # 8. Return answer
    # YOUR CODE HERE
    pass


In [ ]:
# PASS: Checkpoint: TODO 6 — smoke test
_smoke = call_copilot(
    narrative="Smoke test — please ignore. One-line check on Harbor Bridge.",
    inspector_id="smoke_inspector",
    thread_id="copilot_smoke",
    asset_id="Harbor Bridge",
)
assert _smoke and len(_smoke) > 10, "TODO 6 — copilot returned empty/short answer"
print("PASS: TODO 6 passed — copilot ran end-to-end")
print(f"\nSample response (first 300 chars):\n  {_smoke[:300]}")

## The Cross-Inspector Handoff Scenario

Inspector Mercer reviews Harbor Bridge, logs a corrosion finding. Days later, Inspector Vance — who has never met Mercer — encounters a related concern on the same asset. Watch what the copilot tells Vance.

### Day 1 — Mercer's Visit Uses Both Functions, In Sequence

The two functions you built in Parts 4 and 6 play **complementary roles** in a real inspection workflow. Mercer's day-1 visit shows both:

| Step | Function | Role | Cost |
|---|---|---|---|
| 1 | `call_copilot(...)` | Reasoning interface. Mercer types what she sees; the copilot interprets, suggests, surfaces any prior findings on this asset. | ~4 LLM calls, 30–60 s |
| 2 | `log_finding(...)` | System of record. Mercer decides what the finding is and stores it as a structured row with category, severity, recommendation, grade. | 0 LLM calls, ~100 ms |

**Why both?** `call_copilot` is the diagnostic dialogue; `log_finding` is the formal record. The structured row Mercer logs in step 2 becomes **searchable evidence** for the next inspector's `call_copilot` invocation — via `find_similar_findings` inside the copilot — closing the loop.

Below: each function gets its own cell.

In [ ]:
# Step 1 — Mercer reports what she sees and asks the copilot for help.
# This is the REASONING call: builds a context card, vector-searches
# CITY_INSPECTION_FINDING for prior issues on Harbor Bridge, and runs the
# agent LLM. On this first visit there are no prior findings, so the
# copilot's response is mostly fresh diagnosis.
print("=" * 70)
print("DAY 1, step 1: Mercer calls call_copilot for diagnostic help")
print("=" * 70)
MERCER_NOTES = call_copilot(
    narrative=(
        "Quarterly inspection of Harbor Bridge. Surface corrosion observed on Pier 2 "
        "bearing assemblies (south side), estimated section loss ~25% on bearing plate "
        "edges with local rust bleeding onto the concrete pedestal. Corrosion extends "
        "roughly 1.5 m longitudinally along the bearing line."
    ),
    inspector_id="Evelyn_H_Mercer",
    thread_id="asset_harbor_bridge",
    asset_id="Harbor Bridge",
)
print(f"\nMercer's copilot response:\n{MERCER_NOTES}")

Now Mercer has decided this is a real corrosion concern. The copilot's answer helped her think; the structured fields are her conclusion.

**Step 2** is the recording call — `log_finding` writes one row into `CITY_INSPECTION_FINDING` with the embedding computed locally on the `description`. No LLM, no thread, no context card. Cheap and deterministic.

This row is what `find_similar_findings` will surface for the next inspector.

In [ ]:
# Step 2 — Mercer formally records the finding she's now confident about.
# Pure persistence: one INSERT + one local embedding. Fast and free.
MERCER_FINDING_ID = log_finding(
    asset_id="Harbor Bridge",
    inspector="Evelyn_H_Mercer",
    category="corrosion",
    severity="medium",
    description=(
        "Surface corrosion + pitting on steel bearing assemblies at Pier 2 south, "
        "~25% section loss; rust bleed onto concrete pedestal; ~1.5m longitudinal extent."
    ),
    recommendation=(
        "Remove loose corrosion products, apply corrosion-inhibiting primer + finish "
        "coat to affected bearing assemblies within 60 days; re-inspect annually with "
        "caliper section-loss measurements."
    ),
    overall_grade="C",
    days_ago=0,
)
print(f"Logged finding {MERCER_FINDING_ID} — now searchable by future call_copilot invocations.")

In [ ]:
# Day 1, 14:00 — Mercer adds a follow-up observation
print("\n" + "=" * 70)
print("DAY 1 (afternoon): Mercer follow-up note")
print("=" * 70)
MERCER_FOLLOWUP = call_copilot(
    narrative=(
        "Coordinated with maintenance — recommend scheduling the bearing remediation "
        "to coincide with the deck wearing-surface resurfacing in Q3 to share access "
        "and traffic management."
    ),
    inspector_id="Evelyn_H_Mercer",
    thread_id="asset_harbor_bridge",
    asset_id="Harbor Bridge",
)
print(f"\n Mercer's followup response:\n{MERCER_FOLLOWUP}")

### Day N — Vance Only Calls `call_copilot`. Why?

Vance is in the diagnostic phase — he's seen rust bleed and possible spalling, but he hasn't decided what the finding is yet. He calls `call_copilot` to **interpret** what he's seeing, not to record anything.

Under the hood, `call_copilot` does two things that pull in Mercer's prior work without anyone telling it to:

1. **`find_similar_findings(...)`** vector-searches `CITY_INSPECTION_FINDING` for Harbor Bridge — Mercer's logged finding (the row from the cell above) is the top hit. The copilot's prompt gets her category, severity, recommendation, and grade as structured fields.
2. **`thread.get_context_card(...)`** assembles Mercer's earlier messages + any auto-extracted facts/guidelines/preferences from those messages. So Vance gets her conversational reasoning + soft observations (like *coordinate with Q3 deck resurfacing*) that the formal finding doesn't carry.

If Vance later confirms his diagnosis, he'd add a step-2 `log_finding(...)` of his own — but the workshop stops here because the magic moment is Vance's reasoning being shaped by Mercer's prior work without human handoff.

In [ ]:
# Day N (later) — Inspector Vance arrives. Different inspector, same asset.
# Only call_copilot — Vance is still in diagnosis mode, not yet recording.
print("\n" + "=" * 70)
print("DAY N: Inspector Vance on Harbor Bridge (never met Mercer)")
print("=" * 70)
VANCE_DIAGNOSIS = call_copilot(
    narrative=(
        "Reviewing Harbor Bridge as part of routine cycle. Noticing rust bleed near "
        "a pier on the south side and what looks like spalling on the concrete "
        "pedestal below."
    ),
    inspector_id="Jordan_Vance",
    thread_id="asset_harbor_bridge",
    asset_id="Harbor Bridge",
)
print(f"\nVance's copilot response (built from Mercer's work, NO human handoff):\n{VANCE_DIAGNOSIS}")

## Compare: Vance With And Without Memory

Below: same Vance narrative, but stripped of all memory layers — no thread, no context card, no `find_similar_findings`. Just the LLM and the narrative. The contrast is the workshop's point.

In [ ]:
stateless_messages = [
    {"role": "system", "content": COPILOT_SYSTEM_PROMPT},
    {"role": "user",   "content": (
        "Reviewing Harbor Bridge as part of routine cycle. Noticing rust bleed near "
        "a pier on the south side and what looks like spalling on the concrete "
        "pedestal below."
    )},
]
STATELESS_VANCE = call_openai_chat(stateless_messages).choices[0].message.content

print("=" * 70)
print("WITHOUT MEMORY (stateless LLM):")
print("=" * 70)
print(STATELESS_VANCE)
print("\n" + "=" * 70)
print("WITH MEMORY (the copilot you built):")
print("=" * 70)
print(VANCE_DIAGNOSIS)

## Key Takeaways

| Layer | Earned its keep by… |
|---|---|
| Auto-extracted `fact` / `preference` / `guideline` (SDK) | Surfacing tribal knowledge from real maintenance narratives without explicit code |
| `CITY_ASSET` SQL table | Letting the copilot look up structured asset facts at the start of every turn |
| `CITY_INSPECTION_FINDING` SQL with `VECTOR(384)` | Vector-searchable history via Oracle's native `VECTOR_DISTANCE()` — mixed with relational filters in one SQL |
| Context card (SDK) | Compressing the per-asset thread into ~200 tokens that travel with every turn |
| Scoping (`user_id` / `agent_id` / `thread_id`) | Multi-tenant-safety at the SQL layer |

## Production Hardening Checklist

- Real asset registry wired to your CMMS (IBM Maximo, Esri, ProjectMates)
- Asset hierarchy via self-FK on `CITY_ASSET.parent_asset_id`
- Oracle VPD policies on `CITY_MEMORY` for hard row-level security
- Audit log of LLM-generated diagnoses
- Push findings back to inspection-tracking system on confirmation
- Geospatial column (`SDO_GEOMETRY`) for proximity queries
- Periodic re-extraction when domain vocabulary evolves

## Where to Next?

- **[Oracle AI Agent Memory documentation](https://docs.oracle.com/en/database/oracle/agent-memory/)**
- **[Oracle AI Developer Hub](https://github.com/oracle-devrel/oracle-ai-developer-hub)**
- **[Agent Memory short course (DeepLearning.AI)](https://www.deeplearning.ai/short-courses/agent-memory-building-memory-aware-agents/)**